# Fase 1.4 — Tamaño muestral y potencia, antes de modelar

**Tesis v2 · Eider Johan Díaz Berrío · MCDA EAFIT**

Con 21 eventos en la población primaria, la pregunta honesta no es "¿qué modelo gana?" sino **"¿qué se puede afirmar con estos datos?"**. Respondo dos preguntas, ambas antes de ajustar un solo modelo:

1. **¿Cuántos parámetros puede sostener un modelo con datos propios?** Uso los criterios de Riley et al. (2020) para modelos de predicción binarios: contracción esperada ≥ 0,9, optimismo del R² de Nagelkerke ≤ 0,05 y precisión del intercepto de ±0,05.
2. **¿Qué diferencias de AUC son detectables?** Uso el error estándar de Hanley y McNeil (1982) para el AUC de un modelo y una aproximación para la diferencia entre dos modelos correlacionados.

Los resultados alimentan el anexo A del protocolo (`00_protocolo/plan_analisis.md`).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from tesis_v2 import config, estadistica as est, figuras, io_datos

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)
figuras.estilo()

ds = io_datos.leer_parquet(config.PRIVADO / "dataset_analitico_v2.parquet")
lib = ds[ds["producto"] == "libranza"]
pob = {}
for ev in ["y_90_12m", "y_60_12m", "y_30_6m"]:
    mad = "madura_6m" if ev == "y_30_6m" else "madura_12m"
    p = lib[lib[mad] & (lib["cierre_v2"] != "INDETERMINADO") & (lib["anio_cohorte"] != 2021)]
    pob[ev] = (len(p), int(p[ev].sum()))
pd.DataFrame(pob, index=["creditos", "eventos"]).T

,creditos,eventos
y_90_12m,731,21
y_60_12m,731,27
y_30_6m,788,62


## 1. Criterios de Riley et al. (2020)

Los criterios necesitan una anticipación del R² de Cox-Snell del modelo. En lugar de asumir un valor arbitrario, lo derivo del AUC que considero plausible (0,65, 0,70 y 0,75) y de la prevalencia observada, con la simulación propuesta por Riley et al. (2021). Calculo el tamaño mínimo para modelos de 5, 10 y 20 parámetros.

In [2]:
filas = []
for ev, (n, e) in pob.items():
    prev = e / n
    for auc in [0.65, 0.70, 0.75]:
        r2 = est.r2cs_desde_auc(auc, prev)
        for p in [5, 10, 20]:
            r = est.riley_binario(p, prev, r2)
            filas.append({"objetivo": ev, "prevalencia_%": round(100 * prev, 2), "auc_anticipado": auc,
                          "r2cs_anticipado": round(r2, 4), "parametros": p, **r,
                          "n_disponible": n, "eventos_disponibles": e, "cumple": n >= r["n_minimo"]})
riley = pd.DataFrame(filas)
io_datos.guardar_resultado(riley, "fase1", "tamano_muestral_riley")
riley[riley["objetivo"] == "y_90_12m"][["auc_anticipado", "r2cs_anticipado", "parametros", "n_crit1_contraccion",
                                        "n_crit2_optimismo", "n_crit3_intercepto", "n_minimo", "eventos_minimos",
                                        "n_disponible", "cumple"]]

,auc_anticipado,r2cs_anticipado,parametros,n_crit1_contraccion,n_crit2_optimismo,n_crit3_intercepto,n_minimo,eventos_minimos,n_disponible,cumple
0,0.65,0.0082,5,5447,432,43,5447,157,731,False
1,0.65,0.0082,10,10893,864,43,10893,313,731,False
2,0.65,0.0082,20,21786,1727,43,21786,626,731,False
3,0.70,0.0151,5,2957,431,43,2957,85,731,False
4,0.70,0.0151,10,5913,861,43,5913,170,731,False
5,0.70,0.0151,20,11825,1721,43,11825,340,731,False
6,0.75,0.0247,5,1797,428,43,1797,52,731,False
7,0.75,0.0247,10,3593,856,43,3593,104,731,False
8,0.75,0.0247,20,7186,1712,43,7186,207,731,False


¿Cuántos parámetros soportan los datos disponibles? Despejo el criterio 1 (contracción ≥ 0,9), que es el que manda con prevalencias bajas: \(p_{\max} = n\,(S-1)\ln(1 - R^2_{cs}/S)\).

In [3]:
pmax = []
for ev, (n, e) in pob.items():
    prev = e / n
    for auc in [0.65, 0.70, 0.75]:
        r2 = est.r2cs_desde_auc(auc, prev)
        pmax.append({"objetivo": ev, "creditos": n, "eventos": e, "auc_anticipado": auc,
                     "parametros_maximos_S090": round(n * (0.9 - 1) * np.log(1 - r2 / 0.9), 1)})
pmax = pd.DataFrame(pmax)
io_datos.guardar_resultado(pmax, "fase1", "parametros_maximos")
pmax.pivot(index="objetivo", columns="auc_anticipado", values="parametros_maximos_S090")

auc_anticipado,0.65,0.70,0.75
objetivo,,,
y_30_6m,1.9,3.4,5.6
y_60_12m,0.9,1.6,2.6
y_90_12m,0.7,1.2,2.0


**Lectura.** Es el resultado más importante de toda la Fase 1 para el diseño. Con el evento primario y un AUC plausible de 0,65 a 0,75, los datos propios sostienen **entre 0,7 y 2 parámetros** sin una contracción mayor al 10 %. Con 30+/6m se llega a lo sumo a unos 6. Un modelo de 10 parámetros necesitaría entre 3.600 y 10.900 créditos (100 a 300 eventos). Para comparar: el modelo de la v1 usaba unos 45 predictores. Esto no invalida el ejercicio, pero sí obliga a tres cosas, que ya están en el protocolo:

- penalización fuerte y conjuntos de variables muy reducidos;
- información externa (Home Credit) como *prior*, para que el modelo no dependa solo de 21 eventos;
- presentar los resultados como evidencia de factibilidad con incertidumbre explícita, no como un modelo listo para producción.

## 2. Potencia para el AUC

Con el error estándar de Hanley y McNeil calculo el ancho del IC 95 % del AUC para cada objetivo. Para la diferencia entre dos modelos ajustados sobre los mismos créditos uso \(SE_{\Delta} = \sqrt{2\,SE^2(1-r)}\), con una correlación entre AUC de \(r = 0{,}5\) (supuesto moderado y habitual para modelos con variables comunes). La diferencia mínima detectable con potencia del 80 % y \(\alpha = 0{,}05\) es \((z_{0,975} + z_{0,80})\,SE_{\Delta}\).

In [4]:
z = stats.norm.ppf(0.975) + stats.norm.ppf(0.80)
pot = []
for ev, (n, e) in pob.items():
    for auc in [0.60, 0.65, 0.70, 0.75]:
        se = est.se_auc_hanley(auc, e, n - e)
        se_d = np.sqrt(2 * se ** 2 * (1 - 0.5))
        pot.append({"objetivo": ev, "eventos": e, "auc": auc, "se_auc": round(se, 4),
                    "ic95": f"{auc - 1.96 * se:.2f}–{min(auc + 1.96 * se, 1):.2f}",
                    "diferencia_minima_detectable_r05": round(z * se_d, 3),
                    "potencia_vs_azar_%": round(100 * stats.norm.cdf((auc - 0.5) / se - 1.96), 1)})
pot = pd.DataFrame(pot)
io_datos.guardar_resultado(pot, "fase1", "potencia_auc")
pot

,objetivo,eventos,auc,se_auc,ic95,diferencia_minima_detectable_r05,potencia_vs_azar_%
0,y_90_12m,21,0.60,0.0662,0.47–0.73,0.186,32.6
1,y_90_12m,21,0.65,0.0660,0.52–0.78,0.185,62.3
2,y_90_12m,21,0.70,0.0647,0.57–0.83,0.181,87.1
3,y_90_12m,21,0.75,0.0623,0.63–0.87,0.175,98.0
4,y_60_12m,27,0.60,0.0586,0.49–0.71,0.164,40.0
5,y_60_12m,27,0.65,0.0584,0.54–0.76,0.164,72.9
6,y_60_12m,27,0.70,0.0572,0.59–0.81,0.160,93.8
7,y_60_12m,27,0.75,0.0551,0.64–0.86,0.154,99.5
8,y_30_6m,62,0.60,0.0393,0.52–0.68,0.110,72.0
9,y_30_6m,62,0.65,0.0391,0.57–0.73,0.110,97.0


In [5]:
eventos = np.arange(5, 401, 1)
fig, ax = plt.subplots(figsize=(6.6, 3.6))
for auc, col in [(0.65, figuras.AZUL), (0.75, figuras.NARANJA)]:
    prev = pob["y_90_12m"][1] / pob["y_90_12m"][0]
    dmd = [z * np.sqrt(est.se_auc_hanley(auc, int(e_), int(e_ / prev - e_)) ** 2) for e_ in eventos]
    ax.plot(eventos, dmd, color=col, lw=1.8, label=f"AUC de referencia {auc:.2f}")
for ev, mk in [("y_90_12m", "o"), ("y_60_12m", "s"), ("y_30_6m", "^")]:
    e = pob[ev][1]
    d = z * est.se_auc_hanley(0.65, e, pob[ev][0] - e)
    ax.plot(e, d, mk, color="black", ms=6)
    ax.annotate(ev.replace("y_", "").replace("_", " / "), (e, d), xytext=(6, 4), textcoords="offset points", fontsize=8)
ax.axhline(0.05, color=figuras.GRIS, ls="--", lw=0.8)
ax.text(300, 0.058, "diferencia de 0,05", fontsize=7.5, color=figuras.GRIS)
prev90 = pob["y_90_12m"][1] / pob["y_90_12m"][0]
e_005 = next(int(e_) for e_ in range(21, 3000) if z * est.se_auc_hanley(0.65, e_, int(e_ / prev90 - e_)) <= 0.05)
print(f"Eventos necesarios para detectar una diferencia de AUC de 0,05 (AUC 0,65, r = 0,5): {e_005}")
ax.set_xlabel("Número de eventos")
ax.set_ylabel("Diferencia de AUC detectable\n(80 % potencia, r = 0,5)")
ax.set_title("Diferencias de AUC detectables según el número de eventos")
ax.legend()
figuras.guardar(fig, "fase1", "F1_03", "potencia_auc",
                "Con 21 eventos solo se detectan diferencias de AUC del orden de 0,18; para detectar 0,05 harían falta cerca de 300 eventos.")

Eventos necesarios para detectar una diferencia de AUC de 0,05 (AUC 0,65, r = 0,5): 287


**Lectura.** Con 21 eventos, el IC 95 % del AUC de un modelo tiene un ancho de ±0,13, y solo son detectables diferencias entre modelos del orden de **0,18**. Ninguna de las comparaciones de la v1 (por ejemplo 0,658 frente a 0,620) era distinguible. Para detectar diferencias de 0,05 harían falta cerca de 300 eventos, el número exacto impreso arriba. Con 30+/6m (62 eventos) la diferencia detectable baja a 0,11. Esta cifra alimenta la simulación de potencia hacia adelante de la Fase 6: cuántos meses de operación se necesitan para que el componente adaptativo sea evaluable.

In [6]:
r90 = riley[(riley["objetivo"] == "y_90_12m") & (riley["auc_anticipado"] == 0.70)]
anexo = ["", "## Anexo A — Resultados (notebook 04, ejecutado antes de modelar)", "",
         f"Población primaria: {pob['y_90_12m'][0]} créditos y {pob['y_90_12m'][1]} eventos 90+/12m "
         f"(60+/12m: {pob['y_60_12m'][1]} eventos; 30+/6m: {pob['y_30_6m'][1]} eventos en {pob['y_30_6m'][0]} créditos).", "",
         "| Objetivo | AUC anticipado | Parámetros máximos (S ≥ 0,9) |", "|---|---|---|"]
anexo += [f"| {r.objetivo} | {r.auc_anticipado:.2f} | {r.parametros_maximos_S090} |" for r in pmax.itertuples()]
anexo += ["", "Tamaño mínimo (Riley) para 90+/12m con AUC anticipado de 0,70:", "",
          "| Parámetros | n mínimo | eventos mínimos | n disponible |", "|---|---|---|---|"]
anexo += [f"| {r.parametros} | {r.n_minimo} | {r.eventos_minimos} | {r.n_disponible} |" for r in r90.itertuples()]
p90 = pot[pot["objetivo"] == "y_90_12m"]
anexo += ["", "Potencia para el AUC con el evento primario (Hanley y McNeil; diferencia entre modelos con r = 0,5):", "",
          "| AUC | IC 95 % | Diferencia mínima detectable |", "|---|---|---|"]
anexo += [f"| {r.auc:.2f} | {r.ic95} | {r.diferencia_minima_detectable_r05} |" for r in p90.itertuples()]
anexo += ["", "**Implicación para el diseño:** los datos propios sostienen a lo sumo uno o dos parámetros sin "
          "contracción fuerte. Por eso se usan penalización, información previa de Home Credit y objetivos auxiliares, "
          "y las comparaciones entre modelos se leen con sus IC, no por el valor puntual."]
proto = config.RAIZ_V2 / "00_protocolo" / "plan_analisis.md"
texto = proto.read_text(encoding="utf-8")
marca = "## Anexo A — Resultados"
if marca in texto:
    texto = texto[:texto.index(marca)].rstrip() + "\n"
proto.write_text(texto.rstrip() + "\n" + "\n".join(anexo) + "\n", encoding="utf-8")
_ = io_datos.escribir_registro("04_tamano_muestral_y_potencia", {"poblaciones": pob,
                                                                 "salidas": ["resultados/fase1/tamano_muestral_riley.csv",
                                                                             "resultados/fase1/potencia_auc.csv"]})
figuras.escribir_catalogo("fase1", "Figuras de la Fase 1")
print("Anexo A actualizado en el protocolo.")

Anexo A actualizado en el protocolo.


## Conclusiones

- Con 21 eventos, los criterios de Riley indican que los datos propios sostienen **entre 0,7 y 2 parámetros** sin contracción importante. Un modelo de 10 parámetros necesitaría entre 3.600 y 10.900 créditos.
- Las diferencias de AUC detectables son del orden de 0,18. Comparar modelos por su AUC puntual no tiene sustento: la regla de selección del protocolo se basa en *log loss* con IC pareado y prefiere el modelo más simple.
- Esto convierte la limitación en diseño: penalización, *prior* externo y la honestidad de reportar "lo que se puede y lo que no se puede afirmar".